# ETL — Austria Train Data

Ce notebook transforme les données GTFS autrichienne en un format standardisé avec les colonnes :
`data_source`, `route_id`, `id_origin_city`, `id_destination_city`, `weekly_train`, `desserte_type`

**Sources :** `data/Austria/`
- `routes.csv` — infos sur les lignes
- `trips.csv` — association route ↔ trip
- `stop_times.csv` — séquence des arrêts par trip
- `stops.csv` — métadonnées des arrêts

## 0. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../../../data/austria")
DATA_SOURCE = "austria"

## 1. Chargement des fichiers bruts

In [2]:
routes = pd.read_csv(
    DATA_DIR / "routes.csv",
    usecols=["route_id", "route_short_name", "route_long_name", "route_type"],
    dtype=str
)

trips = pd.read_csv(
    DATA_DIR / "trips.csv",
    usecols=["route_id", "trip_id"],
    dtype=str
)

stop_times = pd.read_csv(
    DATA_DIR / "stop_times.csv",
    usecols=["trip_id", "stop_id", "stop_sequence"],
    dtype={"trip_id": str, "stop_id": str, "stop_sequence": int}
)

stops = pd.read_csv(
    DATA_DIR / "stops.csv",
    usecols=["stop_id", "stop_name", "parent_station"],
    dtype=str
)

print(f"routes : {routes.shape}")
print(f"trips : {trips.shape}")
print(f"stop_times : {stop_times.shape}")
print(f"stops : {stops.shape}")

routes : (273, 4)
trips : (15165, 2)
stop_times : (174124, 3)
stops : (7698, 3)


## 2. Extraction des arrêts origine / destination par trip

Pour chaque `trip_id`, on retient :
- **origin** = arrêt avec le `stop_sequence` le plus petit
- **destination** = arrêt avec le `stop_sequence` le plus grand

In [3]:
# Premier arrêt de chaque trip
origin = (
    stop_times
    .sort_values("stop_sequence")
    .groupby("trip_id", as_index=False)
    .first()[["trip_id", "stop_id"]]
    .rename(columns={"stop_id": "id_origin_city"})
)

# Dernier arrêt de chaque trip
destination = (
    stop_times
    .sort_values("stop_sequence")
    .groupby("trip_id", as_index=False)
    .last()[["trip_id", "stop_id"]]
    .rename(columns={"stop_id": "id_destination_city"})
)

od_per_trip = origin.merge(destination, on="trip_id")

# Filtrer les trips sans mouvement (origine == destination)
od_per_trip = od_per_trip[od_per_trip["id_origin_city"] != od_per_trip["id_destination_city"]]

print(f"Trips avec OD valides : {len(od_per_trip):,}")
od_per_trip.head()

Trips avec OD valides : 15,165


,trip_id,id_origin_city,id_destination_city
0,1.TA.1-80-j26-1.1.H,at:49:1349:0:10,at:43:4132:0:1
1,1.TA.1-MB4-j26-1.1.R,at:48:134:0:8,at:48:130:0:4
2,1.TA.1-MS1-V-j26-1.1.R,at:48:130:0:4,at:48:452:0:11
3,1.TA.1-S1-K-j26-1.1.R,at:42:2127:0:1,at:42:2128:0:4
4,1.TA.1-S1-M-j26-1.1.R,at:46:3040:0:10,at:46:2042:0:7


## 3. Jointures : trips → routes + OD

In [4]:
# Associer chaque trip à sa route
trips_enriched = trips.merge(od_per_trip, on="trip_id", how="inner")

# Ajouter les infos de route
trips_enriched = trips_enriched.merge(routes, on="route_id", how="left")

print(f"Lignes après jointures : {len(trips_enriched):,}")
trips_enriched.head()

Lignes après jointures : 15,165


,route_id,trip_id,id_origin_city,id_destination_city,route_short_name,route_long_name,route_type
0,10-A10-1-j26-1,1.TA.10-A10-1-j26-1.1.R,at:42:3654:0:7,at:45:50002:0:4,A10-1,Salzburg - Villach - Klagenfurt,2
1,10-A10-1-j26-1,10.TA.10-A10-1-j26-1.7.H,at:45:50002:0:12,at:49:1349:0:9,A10-1,Salzburg - Villach - Klagenfurt,2
2,10-A10-1-j26-1,11.TA.10-A10-1-j26-1.8.H,at:42:3654:0:34,at:49:1349:0:4,A10-1,Salzburg - Villach - Klagenfurt,2
3,10-A10-1-j26-1,2.TA.10-A10-1-j26-1.2.R,at:42:3654:0:7,at:45:50002:0:5,A10-1,Salzburg - Villach - Klagenfurt,2
4,10-A10-1-j26-1,3.TA.10-A10-1-j26-1.3.R,at:42:3654:0:7,at:45:50002:0:8,A10-1,Salzburg - Villach - Klagenfurt,2


## 4. Calcul de `weekly_train`

On agrège par `(route_id, id_origin_city, id_destination_city)` et on compte les trips distincts.
Ce comptage est un proxy du volume hebdomadaire (les données GTFS représentent typiquement une semaine type).

In [5]:
aggregated = (
    trips_enriched
    .groupby(["route_id", "id_origin_city", "id_destination_city"], as_index=False)
    .agg(weekly_train=("trip_id", "nunique"))
)

print(f"Paires OD uniques : {len(aggregated):,}")
print(f"\nDistribution weekly_train :")
print(aggregated["weekly_train"].describe())

Paires OD uniques : 3,683

Distribution weekly_train :
count    3683.000000
mean        4.117567
std         8.449519
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max       269.000000
Name: weekly_train, dtype: float64


## 5. Calcul de `desserte_type`

| Condition | Valeur |
|---|---|
| `weekly_train < 7` | `Sous-desservi` |
| `7 ≤ weekly_train ≤ 56` | `Desserte Normale` |
| `weekly_train > 56` | `Bien desservi` |

In [6]:
def classify_desserte(n):
    if n < 7:
        return "Sous-desservi"
    elif n <= 56:
        return "Desserte Normale"
    else:
        return "Bien desservi"

aggregated["desserte_type"] = aggregated["weekly_train"].apply(classify_desserte)

print("Répartition desserte_type :")
print(aggregated["desserte_type"].value_counts())

Répartition desserte_type :
desserte_type
Sous-desservi       3132
Desserte Normale     542
Bien desservi          9
Name: count, dtype: int64


## 6. Construction du DataFrame final

In [7]:
OUTPUT_COLS = [
    "data_source",
    "route_id",
    "id_origin_city",
    "id_destination_city",
    "weekly_train",
    "desserte_type",
]

result = aggregated.copy()
result["data_source"] = DATA_SOURCE
result = result[OUTPUT_COLS].sort_values(["route_id", "id_origin_city", "id_destination_city"]).reset_index(drop=True)

print(f"Shape finale : {result.shape}")
print(f"Colonnes : {list(result.columns)}")
result.head(10)

Shape finale : (3683, 6)
Colonnes : ['data_source', 'route_id', 'id_origin_city', 'id_destination_city', 'weekly_train', 'desserte_type']


,data_source,route_id,id_origin_city,id_destination_city,weekly_train,desserte_type
0,austria,1-80-j26-1,at:49:1349:0:10,at:43:4132:0:1,1,Sous-desservi
1,austria,1-MB4-j26-1,at:48:130:0:12,at:48:134:0:8,37,Desserte Normale
2,austria,1-MB4-j26-1,at:48:134:0:8,at:48:130:0:1,1,Sous-desservi
3,austria,1-MB4-j26-1,at:48:134:0:8,at:48:130:0:10,3,Sous-desservi
4,austria,1-MB4-j26-1,at:48:134:0:8,at:48:130:0:4,37,Desserte Normale
5,austria,1-MS1-V-j26-1,at:48:130:0:4,at:48:452:0:11,1,Sous-desservi
6,austria,1-S1-K-j26-1,at:42:2127:0:1,at:42:2128:0:4,1,Sous-desservi
7,austria,1-S1-K-j26-1,at:42:2127:0:1,at:42:3657:0:2,1,Sous-desservi
8,austria,1-S1-K-j26-1,at:42:2127:0:1,at:42:3715:0:1,11,Desserte Normale
9,austria,1-S1-K-j26-1,at:42:2127:0:1,at:42:3715:0:2,2,Sous-desservi


## 7. Contrôles qualité

In [8]:
print("Valeurs nulles")
print(result.isnull().sum())

n_dup = result.duplicated(subset=["route_id", "id_origin_city", "id_destination_city"]).sum()
print(f"{n_dup} doublon(s) détecté(s)")

assert (result["weekly_train"] > 0).all(), "Des weekly_train nuls ou négatifs détectés !"

valid_types = {"Sous-desservi", "Desserte Normale", "Bien desservi"}
assert set(result["desserte_type"].unique()).issubset(valid_types)

Valeurs nulles
data_source            0
route_id               0
id_origin_city         0
id_destination_city    0
weekly_train           0
desserte_type          0
dtype: int64
0 doublon(s) détecté(s)


## 8. Export

In [9]:
OUTPUT_PATH = Path("../../../data/output/austria_etl.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

result.to_csv(OUTPUT_PATH, index=False)
print(f"Export {OUTPUT_PATH.resolve()}")
print(f"   {len(result):,} lignes exportées")

Export C:\Users\kevyn\Documents\Taff\B3\TPRE612\data\output\austria_etl.csv
   3,683 lignes exportées
